# Director-AI — Multi-Provider SDK Guard

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anulum/director-ai/blob/main/notebooks/08_provider_adapters.ipynb)

Director-AI `guard()` auto-detects 5 LLM SDK shapes:
- **OpenAI** (also Groq, vLLM, LiteLLM, Ollama, Together)
- **Anthropic**
- **AWS Bedrock** (boto3 converse API)
- **Google Gemini** (google-generativeai)
- **Cohere** (v2 chat API)

This notebook demonstrates each with mock clients (no API keys needed).

In [ ]:
!pip install -q director-ai

In [ ]:
from types import SimpleNamespace
from director_ai import guard, get_score

## 1. OpenAI-Compatible

Any client with `client.chat.completions.create()`.

In [ ]:
def make_openai_mock():
    resp = SimpleNamespace(
        choices=[SimpleNamespace(message=SimpleNamespace(content="The sky is blue."))]
    )
    completions = SimpleNamespace(create=lambda **kw: resp)
    chat = SimpleNamespace(completions=completions)
    return SimpleNamespace(chat=chat)

client = make_openai_mock()
client = guard(client, facts={"sky": "The sky is blue due to Rayleigh scattering."}, on_fail="metadata")
resp = client.chat.completions.create(messages=[{"role": "user", "content": "What color is the sky?"}])
print(f"Response: {resp.choices[0].message.content}")
print(f"Score:    {get_score().score:.3f}")

## 2. Anthropic

Client with `client.messages.create()` (no `client.chat`).

In [ ]:
def make_anthropic_mock():
    resp = SimpleNamespace(content=[SimpleNamespace(text="The sky is blue.")])
    messages = SimpleNamespace(create=lambda **kw: resp)
    return SimpleNamespace(messages=messages)

client = make_anthropic_mock()
client = guard(client, facts={"sky": "The sky is blue."}, on_fail="metadata")
resp = client.messages.create(messages=[{"role": "user", "content": "Sky color?"}])
print(f"Response: {resp.content[0].text}")
print(f"Score:    {get_score().score:.3f}")

## 3. AWS Bedrock

Client with `client.converse()` and `client.invoke_model()` (boto3 Bedrock Runtime).

In [ ]:
def make_bedrock_mock():
    def converse(**kw):
        return {
            "output": {
                "message": {
                    "content": [{"text": "The sky is blue due to Rayleigh scattering."}]
                }
            }
        }
    return SimpleNamespace(converse=converse, invoke_model=lambda **kw: {}, converse_stream=lambda **kw: {})

client = make_bedrock_mock()
client = guard(client, facts={"sky": "The sky is blue."}, on_fail="metadata")
resp = client.converse(messages=[{"role": "user", "content": [{"text": "Sky color?"}]}])
print(f"Response: {resp['output']['message']['content'][0]['text']}")
print(f"Score:    {get_score().score:.3f}")

## 4. Google Gemini

Client with `client.generate_content()` (google-generativeai GenerativeModel).

In [ ]:
def make_gemini_mock():
    def generate_content(*args, **kw):
        return SimpleNamespace(text="The sky is blue.")
    return SimpleNamespace(generate_content=generate_content)

client = make_gemini_mock()
client = guard(client, facts={"sky": "The sky is blue."}, on_fail="metadata")
resp = client.generate_content("What color is the sky?")
print(f"Response: {resp.text}")
print(f"Score:    {get_score().score:.3f}")

## 5. Cohere

Client with `client.chat()` (Cohere v2 — no `client.completions`).

In [ ]:
def make_cohere_mock():
    def chat(**kw):
        return SimpleNamespace(text="The sky is blue.")
    return SimpleNamespace(chat=chat, chat_stream=lambda **kw: iter([]))

client = make_cohere_mock()
client = guard(client, facts={"sky": "The sky is blue."}, on_fail="metadata")
resp = client.chat(message="What color is the sky?")
print(f"Response: {resp.text}")
print(f"Score:    {get_score().score:.3f}")

## Summary

| Provider | Detection Shape | Proxy |
|----------|----------------|-------|
| OpenAI/Groq/vLLM | `client.chat.completions.create` | Patches completions |
| Anthropic | `client.messages.create` | Patches messages |
| Bedrock | `client.converse` + `client.invoke_model` | Wraps client |
| Gemini | `client.generate_content` | Wraps client |
| Cohere | `client.chat` (no `.completions`) | Wraps client |

All providers support streaming with periodic coherence checks every 8 tokens.